In [1]:
# Load and evaluate
import duckdb
import numpy as np
import pandas as pd
import joblib

model = joblib.load("../models/lgbm_model.joblib")
preds = pd.read_parquet("../data/gold/nyc/model_lgbm_predictions.parquet")

train_preds = preds[preds["split"] == "train"]
test_preds = preds[preds["split"] == "test"]

results = []

def evaluate(name, actual, predicted):
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    missing = np.isnan(predicted)
    if missing.any():
        print(f"[{name}] WARNING: {missing.sum()} listings had no prediction")
    actual, predicted = actual[~missing], predicted[~missing]
    pct_error = np.abs(predicted - actual) / actual * 100
    row = {
        "model": name, "n_evaluated": len(actual),
        "median_abs_pct_error": round(np.median(pct_error), 1),
        "within_20pct": round((pct_error <= 20).mean() * 100, 1),
        "median_abs_error_$": round(np.median(np.abs(predicted - actual)), 1),
    }
    results.append(row)
    return row

evaluate("lightgbm (train)", train_preds["base_price"], train_preds["predicted_base_price"])
evaluate("lightgbm (test)", test_preds["base_price"], test_preds["predicted_base_price"])
pd.DataFrame(results)

,model,n_evaluated,median_abs_pct_error,within_20pct,median_abs_error_$
0,lightgbm (train),17035,17.9,54.1,31.8
1,lightgbm (test),4378,25.1,40.1,46.8


In [2]:
# Which features matter most?
importance = pd.DataFrame({
    "feature": model.feature_name_,
    "importance": model.feature_importances_,
}).sort_values("importance", ascending=False)

print(importance.head(15).to_string(index=False))

            feature  importance
           latitude        1964
          longitude        1785
    amenities_count        1391
host_listings_count        1358
         host_years         857
     rating_overall         825
  reviews_per_month         727
       accommodates         715
     minimum_nights         649
  number_of_reviews         595
           bedrooms         560
               beds         474
          bathrooms         423
          room_type         313
     license_status         240


In [3]:
# Where does it do worst?
test_merged = test_preds.merge(
    duckdb.sql(
        "SELECT listing_id, room_type, stay_type, borough "
        "FROM read_parquet('../data/gold/nyc/model_table.parquet')"
    ).df(),
    on="listing_id",
)
test_merged["pct_error"] = (
    (test_merged["predicted_base_price"] - test_merged["base_price"]).abs()
    / test_merged["base_price"] * 100
)

print(
    test_merged.groupby(["room_type", "stay_type"])["pct_error"]
    .agg(["count", "median"])
    .round(1)
    .sort_values("median", ascending=False)
)

                              count  median
room_type       stay_type                  
Shared room     short stay       10    47.7
Hotel room      monthly stay     21    47.2
Shared room     monthly stay     23    28.0
Entire home/apt monthly stay   2116    26.6
Private room    short stay      633    24.5
Entire home/apt short stay      385    24.3
Private room    monthly stay   1101    23.8
Hotel room      short stay       89    21.5


In [4]:
full_importance = pd.DataFrame({
    "feature": model.feature_name_,
    "importance": model.feature_importances_,
}).sort_values("importance", ascending=False)

print(full_importance.to_string(index=False))

            feature  importance
           latitude        1964
          longitude        1785
    amenities_count        1391
host_listings_count        1358
         host_years         857
     rating_overall         825
  reviews_per_month         727
       accommodates         715
     minimum_nights         649
  number_of_reviews         595
           bedrooms         560
               beds         474
          bathrooms         423
          room_type         313
     license_status         240
  host_is_superhost         191
        shared_bath         136
            borough         108
          stay_type          39


In [5]:
counts = duckdb.sql(f"""
    SELECT stay_type, COUNT(*) AS listings
    FROM read_parquet('../data/gold/nyc/model_table.parquet')
    GROUP BY stay_type
""").df()
print(counts)

# Also confirm it survived the category conversion correctly
print(model.pandas_categorical)

      stay_type  listings
0    short stay      4993
1  monthly stay     16521


AttributeError: 'LGBMRegressor' object has no attribute 'pandas_categorical'

In [6]:
# How separated are short-stay vs monthly-stay listings, geographically?
geo_by_stay = duckdb.sql(f"""
    SELECT
        stay_type,
        ROUND(AVG(latitude), 4)  AS avg_lat,
        ROUND(AVG(longitude), 4) AS avg_lon,
        ROUND(STDDEV(latitude), 4)  AS std_lat,
        ROUND(STDDEV(longitude), 4) AS std_lon,
        COUNT(*) AS listings
    FROM read_parquet('../data/gold/nyc/model_table.parquet')
    GROUP BY stay_type
""").df()
print(geo_by_stay)

      stay_type  avg_lat  avg_lon  std_lat  std_lon  listings
0    short stay  40.7250 -73.9490   0.0531   0.0628      4993
1  monthly stay  40.7289 -73.9413   0.0604   0.0607     16521


In [7]:
import lightgbm as lgb
from sklearn.model_selection import GroupShuffleSplit

NUMERIC_FEATURES = [
    "accommodates", "bedrooms", "beds", "bathrooms",
    "amenities_count", "minimum_nights", "rating_overall",
    "number_of_reviews", "reviews_per_month",
    "host_listings_count", "host_years",
    "latitude", "longitude",
]
CATEGORICAL_FEATURES = [
    "room_type", "stay_type", "license_status",
    "borough", "host_is_superhost", "shared_bath",
]
TARGET = "log_base_price"

data = duckdb.sql("""
    SELECT m.*, s.split
    FROM read_parquet('../data/gold/nyc/model_table.parquet') m
    JOIN read_parquet('../data/gold/nyc/split.parquet') s
      ON m.listing_id = s.listing_id
""").df()
train = data[data["split"] == "train"].reset_index(drop=True)
test = data[data["split"] == "test"].reset_index(drop=True)

splitter = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
fit_idx, valid_idx = next(splitter.split(train, groups=train["host_id"]))
fit_set = train.iloc[fit_idx].reset_index(drop=True)
valid_set = train.iloc[valid_idx].reset_index(drop=True)

def frame_without_stay(df):
    cols = [c for c in NUMERIC_FEATURES + CATEGORICAL_FEATURES if c != "stay_type"]
    frame = df[cols].copy()
    for col in CATEGORICAL_FEATURES:
        if col in frame.columns:
            frame[col] = frame[col].astype("category")
    return frame

model_no_stay = lgb.LGBMRegressor(
    n_estimators=2000, learning_rate=0.03, num_leaves=31,
    min_child_samples=20, random_state=42, verbose=-1,
)
model_no_stay.fit(
    frame_without_stay(fit_set), fit_set[TARGET],
    eval_set=[(frame_without_stay(valid_set), valid_set[TARGET])],
    callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)],
)
print(f"Stopped after {model_no_stay.best_iteration_} trees")

pred_no_stay = 10 ** model_no_stay.predict(frame_without_stay(test))
evaluate("lightgbm, no stay_type (test)", test["base_price"], pred_no_stay)

pd.DataFrame(results)

c:\Users\win-11\Desktop\RateRight\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Stopped after 442 trees


,model,n_evaluated,median_abs_pct_error,within_20pct,median_abs_error_$
0,lightgbm (train),17035,17.9,54.1,31.8
1,lightgbm (test),4378,25.1,40.1,46.8
2,"lightgbm, no stay_type (test)",4378,25.3,40.6,46.7


In [8]:
segment_counts = duckdb.sql("""
    SELECT m.room_type, m.stay_type, COUNT(*) AS training_listings
    FROM read_parquet('../data/gold/nyc/model_table.parquet') m
    JOIN read_parquet('../data/gold/nyc/split.parquet') s
      ON m.listing_id = s.listing_id
    WHERE s.split = 'train'
    GROUP BY m.room_type, m.stay_type
    ORDER BY training_listings
""").df()

print(segment_counts)

MIN_TRAINING_LISTINGS = 30
print()
print("Segments below threshold:")
print(segment_counts[segment_counts["training_listings"] < MIN_TRAINING_LISTINGS])

         room_type     stay_type  training_listings
0      Shared room    short stay                 30
1      Shared room  monthly stay                127
2       Hotel room    short stay                370
3  Entire home/apt    short stay               1202
4     Private room    short stay               2264
5     Private room  monthly stay               5250
6  Entire home/apt  monthly stay               7792

Segments below threshold:
Empty DataFrame
Columns: [room_type, stay_type, training_listings]
Index: []


In [11]:
def prepare_sql(source: str) -> str:
    k = OUTLIER_IQR_MULTIPLIER
    min_training = MIN_TRAINING_LISTINGS   # new constant, defined near the top of the file
    return f"""
    WITH base AS ( ... ),          -- unchanged
    with_log AS ( ... ),           -- unchanged
    segment_range AS ( ... ),      -- unchanged

    -- how many non-outlier, priced listings exist in each room+stay segment
    segment_size AS (
        SELECT room_type, stay_type, COUNT(*) AS segment_listings
        FROM with_log
        WHERE base_price IS NOT NULL AND NOT price_outlier
        GROUP BY room_type, stay_type
    )

    SELECT
        w.*,
        CASE
            WHEN w.log_base_price IS NULL THEN NULL
            ELSE w.log_base_price > s.q3 + {k} * (s.q3 - s.q1)
              OR w.log_base_price < s.q1 - {k} * (s.q3 - s.q1)
        END AS price_outlier,

        CASE
            WHEN COALESCE(sz.segment_listings, 0) < {min_training} THEN 'low'
            ELSE 'normal'
        END AS price_confidence
    FROM with_log w
    LEFT JOIN segment_range s
      ON w.room_type = s.room_type AND w.stay_type = s.stay_type
    LEFT JOIN segment_size sz
      ON w.room_type = sz.room_type AND w.stay_type = sz.stay_type
    """

In [12]:
OUTLIER_IQR_MULTIPLIER = 3
MIN_TRAINING_LISTINGS = 30   # segments with fewer priced, non-outlier listings get price_confidence = "low"